# Full Model CPT - 8-Bit Quantization (No LoRA)
## Qwen2.5-7B - Complete Model Training

- Trains the FULL model (all 7.62B parameters)
- 8-bit quantization (model: 15GB → 4GB)
- NO adapters, NO LoRA
- Fits on 31.84GB GPU
- Medical domain knowledge + instruction following preserved

In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['CUDA_LAUNCH_BLOCKING'] = '0'

import torch
import json
from pathlib import Path
from datetime import datetime
from typing import Dict, List
import warnings
warnings.filterwarnings('ignore')

print("\n" + "="*80)
print("FULL MODEL CPT - 8-BIT QUANTIZATION (NO LORA)")
print("="*80)
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
print("Memory optimizations: ENABLED")
print("="*80 + "\n")

In [ ]:
# ============ ULTRA-MINIMAL CONFIG ============
config = {
    "train_file": "pretraining_augmented_data/train.jsonl",
    "eval_file": "pretraining_augmented_data/eval.jsonl",
    "num_train_epochs": 3,
    "per_device_train_batch_size": 1,      # ABSOLUTE MINIMUM
    "per_device_eval_batch_size": 1,
    "gradient_accumulation_steps": 4,      # Reduced to minimize thrashing
    "learning_rate": 2e-5,
    "warmup_steps": 100,
    "weight_decay": 0.01,
    "max_grad_norm": 1.0,
    "max_seq_length": 256,                 # REDUCED
    "output_dir": "medical_qwen_cpt_8bit",
    "save_steps": 50,
    "eval_steps": 25,
    "logging_steps": 5,
    "dataloader_num_workers": 0,
    "dataloader_pin_memory": False,
    "seed": 42,
}

print("8-Bit Full Model Configuration:")
print("="*70)
print(f"Model: Qwen2.5-7B (local)")
print(f"Quantization: 8-bit (model 15GB → 4GB)")
print(f"Training: ALL 7.62B parameters")
print(f"Batch size: {config['per_device_train_batch_size']}")
print(f"Gradient accumulation: {config['gradient_accumulation_steps']}")
print(f"Effective batch: {config['per_device_train_batch_size'] * config['gradient_accumulation_steps']}")
print(f"Sequence length: {config['max_seq_length']}")
print(f"Optimizer: paged_adamw_8bit (compressed state)")
print("="*70 + "\n")

In [ ]:
# ============ LOAD DATA ============
def load_jsonl(file_path: str) -> List[Dict]:
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            try:
                data.append(json.loads(line))
            except:
                pass
    return data

print("Loading data...")
train_data = load_jsonl(config["train_file"])
eval_data = load_jsonl(config["eval_file"])

print(f"✅ Train: {len(train_data):,} chunks")
print(f"✅ Eval:  {len(eval_data):,} chunks\n")

In [ ]:
# ============ LOAD TOKENIZER ============
from transformers import AutoTokenizer

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    "Qwen2.5-7B",
    trust_remote_code=True,
    use_fast=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"✅ Tokenizer loaded\n")

In [ ]:
# ============ LOAD MODEL IN 8-BIT (NO LORA) ============
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training

print("Loading model in 8-bit (full parameters, no adapters)...")
print("(This reduces model from 15GB → 4GB)\n")

# Configure 8-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    bnb_8bit_compute_dtype=torch.bfloat16,
    bnb_8bit_use_double_quant=True,
)

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

# Load model from local path
model = AutoModelForCausalLM.from_pretrained(
    "Qwen2.5-7B",
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

# CRITICAL: Prepare for training (unfreezes all parameters)
# This allows full model training without LoRA
model = prepare_model_for_kbit_training(model)

# Enable gradient checkpointing
model.gradient_checkpointing_enable()

print(f"✅ Model loaded")
num_params = sum(p.numel() for p in model.parameters())
print(f"   Parameters: {num_params/1e9:.2f}B")
print(f"   Quantization: 8-bit (compute in bfloat16)")
print(f"   LoRA: NO (training full model)")
print(f"   Gradient checkpointing: ENABLED")

allocated = torch.cuda.memory_allocated(0) / 1e9
total = torch.cuda.get_device_properties(0).total_memory / 1e9
free = total - allocated

print(f"\n   GPU allocated: {allocated:.2f} / {total:.2f} GB")
print(f"   GPU free: {free:.2f} GB")

if free < 10:
    print(f"\n⚠️  WARNING: Only {free:.2f}GB free - monitor carefully!")
else:
    print(f"\n✅ Good: {free:.2f}GB free\n")

In [ ]:
# ============ TOKENIZE ============
from datasets import Dataset

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=config["max_seq_length"],
        padding="max_length",
    )

print("Tokenizing...")
train_dataset = Dataset.from_dict({"text": [c["text"] for c in train_data]})
train_dataset = train_dataset.map(
    tokenize_function,
    batched=True,
    batch_size=100,
    remove_columns=["text"],
)

eval_dataset = Dataset.from_dict({"text": [c["text"] for c in eval_data]})
eval_dataset = eval_dataset.map(
    tokenize_function,
    batched=True,
    batch_size=100,
    remove_columns=["text"],
)

print(f"✅ Train: {len(train_dataset):,} samples")
print(f"✅ Eval: {len(eval_dataset):,} samples\n")

In [ ]:
# ============ SETUP TRAINER ============
from transformers import DataCollatorForLanguageModeling, TrainingArguments, Trainer

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

training_args = TrainingArguments(
    output_dir=config["output_dir"],
    num_train_epochs=config["num_train_epochs"],
    per_device_train_batch_size=config["per_device_train_batch_size"],
    per_device_eval_batch_size=config["per_device_eval_batch_size"],
    gradient_accumulation_steps=config["gradient_accumulation_steps"],
    learning_rate=config["learning_rate"],
    warmup_steps=config["warmup_steps"],
    weight_decay=config["weight_decay"],
    max_grad_norm=config["max_grad_norm"],
    optim="paged_adamw_8bit",
    bf16=True,
    save_strategy="steps",
    save_steps=config["save_steps"],
    save_total_limit=1,
    eval_strategy="steps",
    eval_steps=config["eval_steps"],
    logging_dir=config["output_dir"],
    logging_steps=config["logging_steps"],
    seed=config["seed"],
    dataloader_num_workers=config["dataloader_num_workers"],
    dataloader_pin_memory=config["dataloader_pin_memory"],
    load_best_model_at_end=True,
    greater_is_better=False,
    push_to_hub=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
)

print("✅ Trainer ready\n")
print("="*80)
print("🚀 STARTING FULL MODEL 8-BIT TRAINING")
print("="*80)
print(f"Model: Qwen2.5-7B (8-bit, full params, no LoRA)")
print(f"Data: pretraining_augmented_data")
print(f"Start: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*80 + "\n")

In [ ]:
# ============ TRAIN ============
print(f"Starting training...\n")

try:
    train_result = trainer.train()
    print(f"\n✅ Training complete: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"Training loss: {train_result.training_loss:.4f}\n")
except KeyboardInterrupt:
    print("\n⏹️  Training interrupted by user")
except Exception as e:
    print(f"\n❌ Error during training: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
# ============ EVAL & SAVE ============
print("Evaluating...")
eval_results = trainer.evaluate()
print(f"Eval loss: {eval_results.get('eval_loss', 'N/A'):.4f}")

best_model_path = Path(config["output_dir"]) / "best_model"
best_model_path.mkdir(parents=True, exist_ok=True)

print(f"\nSaving to {best_model_path}...")
trainer.model.save_pretrained(str(best_model_path))
tokenizer.save_pretrained(str(best_model_path))

print("✅ DONE")
print(f"\nModel saved to: {best_model_path}")
print(f"Training finished: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")